# 03. 거시경제 변수 수집

**저장 위치**: `data/raw/macro_data.csv`

| 변수 | 출처 | 설명 |
|------|------|------|
| `dxy` | Yahoo Finance (DX-Y.NYB) | 달러 인덱스 - 달러 강세 시 크립토 자금 이탈 |
| `vix` | Yahoo Finance (^VIX) | 시장 공포지수 - 30 초과 시 극단적 공포 |
| `federal_funds_rate` | FRED (무료, 인증 불필요) | 미국 기준금리 - 금리 인상 시 위험자산 회피 |
| `btc_dominance` | CoinMarketCap 내부 API | BTC 시총 비중 - 하락 시 알트코인/스테이블코인으로 자금 이동 |

In [ ]:
import requests
import pandas as pd
import yfinance as yf
import time
import os
from datetime import datetime
from io import StringIO

SAVE_DIR   = os.path.join('..', 'data', 'raw')
START_DATE = '2019-11-22'
END_DATE   = datetime.today().strftime('%Y-%m-%d')

In [ ]:
def collect_yfinance(ticker, col_name):
    """
    Yahoo Finance에서 일별 종가 수집
    - DXY, VIX는 주말/공휴일 데이터 없음 → 전처리에서 ffill 처리 예정
    """
    print(f'  수집 중: {col_name} ({ticker})...')
    df = yf.download(ticker, start=START_DATE, progress=False)

    # 멀티인덱스 컬럼 처리
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]

    # 종가(Close)만 사용, 컬럼명을 변수명으로 변경
    df = df[['Close']].rename(columns={'Close': col_name})
    df.index.name = 'Date'
    df = df.reset_index()
    df['Date'] = df['Date'].astype(str).str[:10]
    print(f'    완료: {len(df)}개 행')
    return df


def collect_fred_ffr():
    """
    FRED API로 미국 기준금리(Federal Funds Rate) 수집
    - 완전 무료, API 키 불필요
    - 월별 발표 데이터 → 일별로 forward fill (발표 이후 다음 발표까지 동일값 유지)
    - FEDFUNDS: 실효연방기금금리 시리즈 코드
    """
    print('  수집 중: Federal Funds Rate (FRED)...')
    url = 'https://fred.stlouisfed.org/graph/fredgraph.csv'
    res = requests.get(url, params={'id': 'FEDFUNDS'}, timeout=30)
    res.raise_for_status()

    df = pd.read_csv(StringIO(res.text))
    df.columns = ['Date', 'federal_funds_rate']
    df = df[df['Date'] >= START_DATE].reset_index(drop=True)

    # 월별 → 일별 확장: 전체 날짜 범위 생성 후 left join + ffill
    df['Date'] = pd.to_datetime(df['Date'])
    date_range = pd.date_range(start=START_DATE, end=END_DATE, freq='D')
    df_daily = pd.DataFrame({'Date': date_range})
    df_daily = df_daily.merge(df, on='Date', how='left')
    df_daily['federal_funds_rate'] = df_daily['federal_funds_rate'].ffill()
    df_daily['Date'] = df_daily['Date'].astype(str).str[:10]
    print(f'    완료: {len(df_daily)}개 행')
    return df_daily


def collect_btc_dominance():
    """
    CoinMarketCap 내부 API로 BTC 도미넌스(시장 내 BTC 비중) 수집
    - global-metrics: 전체 크립토 시장 지표 API
    - BTC 도미넌스 하락 = 알트코인/스테이블코인으로 자금 분산 신호
    """
    print('  수집 중: BTC 도미넌스 (CoinMarketCap)...')
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'application/json'
    }

    rows = []
    start   = datetime(2019, 11, 22)
    end     = datetime.today()
    chunk   = 180
    current = start

    while current < end:
        chunk_end = min(current + pd.Timedelta(days=chunk), end)
        try:
            res = requests.get(
                'https://api.coinmarketcap.com/data-api/v3/global-metrics/quotes/historical',
                headers=headers,
                params={
                    'convertId': '2781',           # USD
                    'timeStart': int(current.timestamp()),
                    'timeEnd':   int(chunk_end.timestamp()),
                    'interval':  '1d'
                },
                timeout=30
            )
            data = res.json()
            for q in data.get('data', {}).get('quotes', []):
                rows.append({
                    'Date':          q['timestamp'][:10],
                    'btc_dominance': q.get('btcDominance')  # BTC 시총 점유율 (%)
                })
            print(f'    {current.strftime("%Y-%m-%d")} ~ {chunk_end.strftime("%Y-%m-%d")}: {len(data.get("data",{}).get("quotes",[]))}개')
        except Exception as e:
            print(f'    오류: {e}')
        current = chunk_end + pd.Timedelta(days=1)
        time.sleep(1.5)

    df = pd.DataFrame(rows).drop_duplicates(subset='Date').sort_values('Date').reset_index(drop=True)
    print(f'    완료: {len(df)}개 행')
    return df

In [ ]:
# 전체 수집 실행
os.makedirs(SAVE_DIR, exist_ok=True)
dfs = []

print('1. DXY (달러 인덱스)')
dfs.append(collect_yfinance('DX-Y.NYB', 'dxy'))
time.sleep(1)

print('2. VIX (공포지수)')
dfs.append(collect_yfinance('^VIX', 'vix'))
time.sleep(1)

print('3. 미국 기준금리 (FRED)')
dfs.append(collect_fred_ffr())
time.sleep(1)

print('4. BTC 도미넌스 (CoinMarketCap)')
dfs.append(collect_btc_dominance())

In [ ]:
# 4개 변수 병합 (Date 기준 outer join)
merged = dfs[0]
for df in dfs[1:]:
    merged = merged.merge(df, on='Date', how='outer')
merged = merged.sort_values('Date').reset_index(drop=True)
merged = merged[merged['Date'] >= START_DATE].reset_index(drop=True)

save_path = os.path.join(SAVE_DIR, 'macro_data.csv')
merged.to_csv(save_path, index=False)
print(f'저장 완료: {save_path}')
print(f'총 {len(merged)}개 행, {len(merged.columns)}개 컬럼')
merged.head()